# 🐍 Join Datasets Notebook Overview
This notebook processes and merges IMDb and TMDB datasets, applying filtering and cleanup.

## 📂 Steps in This Notebook
1️⃣ **Unpacked IMDb Data** → Extracted GZIP files (one-time setup).  
2️⃣ **Loaded IMDb Title Basics** → Dropped unnecessary columns for efficiency.  
3️⃣ **Merged IMDb with Ratings** → Filtered for movies with **>1,000 votes**.  
4️⃣ **Merged Subgenres** → ⚠️ *Issue:* *Gladiator (2000)* got TMDB data from *Gladiator (1992)* (Needs Fixing).  
5️⃣ **Exported Titles for TMDB Data Retrieval** → Processed in `TMDBDataExtractor.ipynb`.  
6️⃣ **Imported & Merged TMDB Data** → Combined IMDb & TMDB metadata.  
7️⃣ **Final Filtering** → Kept movies with **revenue > $500K** (from **11,072 → 6,028** movies).  

## 📊 Current Dataset Columns
`tconst`, `title`, `year`, `runtime_minutes`, `genres`, `rating`, `numVotes`, `subgenres`, `budget`,  
`TMDB_id`, `origin_country`, `language`, `revenue`, `keywords`, `production_companies`, `cast`, `crew`.

## 🛠️ Next Steps
✔ **Fix *Gladiator* subgenre mismatch**  
✔ **Drop redundant columns (`title_x`, `title_y`, `genres_x`, `genres_y`)**  
✔ **Prepare dataset for graph-building & analysis**  

---

### Unpack (skip this)

In [25]:
import gzip, shutil
import pandas as pd

with gzip.open('IMDb/new/title.basics.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.basics.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.ratings.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.ratings.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/name.basics.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/name.basics.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.akas.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.akas.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open('IMDb/new/title.crew.tsv.gz', 'rb') as f_in:
    with open('IMDb/new/title.crew.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

### Load title_basics data
then filter by year to reduce rows

In [53]:
import pandas as pd

df_title_basics = pd.read_csv('IMDb/new/title.basics.tsv', sep='\t', low_memory=False, na_values=['\\N'])
df_title_basics = df_title_basics.rename(columns={"startYear": "year"})

# Keep only rows where 'runtime' contains valid integers
df_title_basics["runtimeMinutes"] = df_title_basics["runtimeMinutes"].fillna("")
df_filtered = df_title_basics[df_title_basics["runtimeMinutes"].str.isdigit()].copy()
df_filtered["runtimeMinutes"] = df_filtered["runtimeMinutes"].astype(int)

df_title_basics_filtered = df_filtered[(df_filtered["year"] >= 1970) &
                                           (df_filtered["titleType"] == "movie") &
                                           (df_filtered["runtimeMinutes"] >= 60) &
                                           (df_filtered["runtimeMinutes"] <= 300)]

df_title_basics_filtered = df_title_basics_filtered.drop(columns=["isAdult", "endYear", "titleType"])
df_title_basics_filtered["year"] = df_title_basics_filtered["year"].astype(int)

df_title_basics_filtered.head()

,tconst,primaryTitle,originalTitle,year,runtimeMinutes,genres
15479,tt0015724,Dama de noche,Dama de noche,1993,102,"Drama,Mystery,Romance"
34794,tt0035423,Kate & Leopold,Kate & Leopold,2001,118,"Comedy,Fantasy,Romance"
35957,tt0036606,"Another Time, Another Place","Another Time, Another Place",1983,118,"Drama,War"
38749,tt0039442,"Habla, mudita","Habla, mudita",1973,88,Drama
44149,tt0044952,Nagarik,Nagarik,1977,127,Drama


### MERGE BASICS WITH RATINGS
then filter by numVotes to further filter

In [77]:
df_title_ratings = pd.read_csv('IMDb/new/title.ratings.tsv', sep='\t', low_memory=False, na_values=['\\N'])

df_merged = pd.merge(df_title_basics_filtered, df_title_ratings, on="tconst", how="inner")

# FILTER FOR POPULAR MOVIES
df_merged_filtered = df_merged[df_merged["numVotes"] > 1000]

print(f"Rows: {len(df_merged_filtered)}")
df_merged_filtered.head()

Rows: 39339


,tconst,primaryTitle,originalTitle,year,runtimeMinutes,genres,averageRating,numVotes
1,tt0035423,Kate & Leopold,Kate & Leopold,2001,118,"Comedy,Fantasy,Romance",6.4,91466
6,tt0054724,I Eat Your Skin,Zombie,1971,92,Horror,3.6,1733
24,tt0061592,Doomsday Machine,Doomsday Machine,1976,83,Sci-Fi,2.6,1450
30,tt0062690,The Awakening of the Beast,O Ritual dos Sádicos,1970,93,"Drama,Horror",5.9,1368
43,tt0063142,Isle of the Snake People,La muerte viviente,1971,90,"Horror,Mystery",3.4,1095


In [83]:
df_merged_filtered[df_merged_filtered["primaryTitle"] == "Bolero"]

,tconst,primaryTitle,originalTitle,year,runtimeMinutes,genres,averageRating,numVotes
11255,tt0083260,Bolero,Les uns et les autres,1981,173,"Drama,Music",7.3,2894
13359,tt0086987,Bolero,Bolero,1984,105,"Comedy,Drama,Romance",3.0,6238


### MERGE THOSE WITH SUB-GENRES

In [85]:
import ast

df_subgenres = pd.read_csv("IMDb/titles_subgenres.csv")
df_merged2 = df_subgenres.merge(df_merged_filtered, how="inner", left_on="title", right_on="primaryTitle")

# CLEAN
df_merged2["subgenres"] = df_merged2["subgenres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_merged2["subgenres_str"] = df_merged2["subgenres"].apply(lambda x: ", ".join(x))
df_merged2 = df_merged2.drop(columns=["subgenres", "primaryTitle"])
df_merged2 = df_merged2.rename(columns={"subgenres_str": "subgenres"})

print(f"Rows: {len(df_merged2)}")
df_merged2.head()

Rows: 11083


,title,tconst,originalTitle,year,runtimeMinutes,genres,averageRating,numVotes,subgenres
0,Captain America: Brave New World,tt14513804,Captain America: Brave New World,2025,118,"Action,Adventure,Sci-Fi",5.9,75784,"action-epic, epic-adventure, epic-sci-fi, supe..."
1,Gladiator II,tt9218128,Gladiator II,2024,148,"Action,Adventure,Drama",6.6,215175,"action-epic, epic-adventure, epic-drama, perio..."
2,Furiosa: A Mad Max Saga,tt12037194,Furiosa: A Mad Max Saga,2024,148,"Action,Adventure,Sci-Fi",7.5,279232,"action-epic, car-action, desert-adventure, dys..."
3,Dune: Part Two,tt15239678,Dune: Part Two,2024,166,"Action,Adventure,Drama",8.5,611472,"action-epic, desert-adventure, epic-drama, epi..."
4,Gladiator,tt0104346,Gladiator,1992,101,"Action,Drama,Sport",6.5,11092,"action-epic, epic-adventure, epic-drama, perio..."


### EXPORT TITLES FOR FURTHER DATA AQUISITION FROM TMDB

In [87]:
df_merged2[["title", "tconst"]].to_csv("titles_with_subgenres.csv")
print("saved")

saved


### IMPORT JSON DATA

In [118]:
df_tmdb = pd.read_csv("parsed_json_data.csv")
df_tmdb = df_tmdb.rename(columns={"id": "tmdb_id"})

print(f"Rows: {len(df_tmdb)}")
df_tmdb.head()

Rows: 11072


,budget,tmdb_id,origin_country,original_language,original_title,revenue,title,keywords,production_companies,genres,cast,crew,tconst
0,48000000,11232,['US'],en,Kate & Leopold,76019048,Kate & Leopold,"['new york city', 'time travel', 'duke', 'fish...","['Konrad Pictures', 'Miramax']","['Romance', 'Comedy', 'Fantasy']","[{'name': 'Meg Ryan', 'order': 0, 'character':...","[{'name': 'James Mangold', 'job': 'Director'},...",tt0035423
1,0,2912,['FR'],fr,Le Boucher,0,The Butcher,"['vietnam veteran', 'province', 'butcher', 'mu...","['Les Films La Boétie', 'Euro International Fi...","['Thriller', 'Crime', 'Drama']","[{'name': 'Stéphane Audran', 'order': 0, 'char...","[{'name': 'Claude Chabrol', 'job': 'Director'}...",tt0064106
2,0,10237,['US'],en,The Honeymoon Killers,0,The Honeymoon Killers,"['nurse', 'widow', 'alabama', 'lonely hearts a...","['Roxanne Company', 'American International Pi...","['Crime', 'Drama', 'Romance', 'Thriller']","[{'name': 'Shirley Stoler', 'order': 0, 'chara...","[{'name': 'Leonard Kastle', 'job': 'Director'}...",tt0064437
3,0,96243,['GB'],en,I Start Counting,0,I Start Counting,"['exploitation', 'stalker', 'serial killer', '...","['Triumvirate Films', 'United Artists']","['Thriller', 'Drama']","[{'name': 'Jenny Agutter', 'order': 0, 'charac...","[{'name': 'David Greene', 'job': 'Director'}, ...",tt0064462
4,0,62843,['XC'],cs,Kladivo na čarodějnice,0,Witchhammer,"['witch', 'based on novel or book', 'witch bur...",['Filmové studio Barrandov'],"['Drama', 'Thriller']","[{'name': 'Elo Romančík', 'order': 0, 'charact...","[{'name': 'Otakar Vávra', 'job': 'Director'}, ...",tt0064546


### MERGE IMDB & TMDB DATA (CROSSING THE STREAMS)

In [123]:
def merge_tmdb_with_imdb(df_imdb, df_tmdb):
    """Merge TMDB JSON data into the IMDb dataset."""
    # Merge IMDb dataset with extracted TMDB data
    df_merged = pd.merge(df_imdb, df_tmdb, on="tconst")
    
    return df_merged

# Merge TMDB data into IMDb dataset (df_merged2)
df_final = merge_tmdb_with_imdb(df_merged2, df_tmdb)

# Save to CSV or Parquet
#df_final.to_csv("merged_imdb_tmdb.csv", index=False)

df_final.head(2)

,title_x,tconst,originalTitle,year,runtimeMinutes,genres_x,averageRating,numVotes,subgenres,budget,...,origin_country,original_language,original_title,revenue,title_y,keywords,production_companies,genres_y,cast,crew
0,Captain America: Brave New World,tt14513804,Captain America: Brave New World,2025,118,"Action,Adventure,Sci-Fi",5.9,75784,"action-epic, epic-adventure, epic-sci-fi, supe...",180000000,...,['US'],en,Captain America: Brave New World,388056272,Captain America: Brave New World,"['hero', 'superhero', 'revenge', 'aftercredits...","['Marvel Studios', 'Kevin Feige Productions']","['Action', 'Thriller', 'Science Fiction']","[{'name': 'Anthony Mackie', 'order': 0, 'chara...","[{'name': 'Kevin Feige', 'job': 'Producer'}, {..."
1,Gladiator II,tt9218128,Gladiator II,2024,148,"Action,Adventure,Drama",6.6,215175,"action-epic, epic-adventure, epic-drama, perio...",310000000,...,['US'],en,Gladiator II,458718000,Gladiator II,"['epic', 'gladiator', 'roman empire', 'ancient...","['Paramount Pictures', 'Scott Free Productions...","['Action', 'Adventure', 'Drama']","[{'name': 'Paul Mescal', 'order': 0, 'characte...","[{'name': 'John Mathieson', 'job': 'Director o..."


### FILTER

In [130]:
rows_after = df_final[df_final["revenue"] >= 500_000].shape[0]
rows_before = df_final.shape[0]
print(f"Before: {rows_before}\nAfter: {rows_after}")

Before: 11072
After: 6028


In [131]:
df_final.columns

Index(['title_x', 'tconst', 'originalTitle', 'year', 'runtimeMinutes',
       'genres_x', 'averageRating', 'numVotes', 'subgenres', 'budget',
       'tmdb_id', 'origin_country', 'original_language', 'original_title',
       'revenue', 'title_y', 'keywords', 'production_companies', 'genres_y',
       'cast', 'crew'],
      dtype='object')